# Getting nse symbols

In [1]:
# Required for Notebooks
import sys
sys.path.append("C:/Users/kashi/python/ib/src")

# Logger
import logging
import yaml

with open('./config/log.yml', 'r') as f:
    config = yaml.safe_load(f.read())
    logging.config.dictConfig(config)

log = logging.getLogger('ib_log')

In [2]:
# get the symbols
from nse.nse import get_nse_syms
df_syms = get_nse_syms()
df_syms.head(5)

,symbol,secType,exchange,mar-23,apr-23,may-23
0,BANKNIFTY,IND,NSE,25,25,25
1,NIFTY,IND,NSE,50,50,50
2,MIDCPNIFTY,IND,NSE,75,75,75
3,FINNIFTY,IND,NSE,40,40,40
5,ABCAPITAL,STK,NSE,5400,5400,5400


# Build Option Chains

In [5]:
# get the chains
import tqdm
from nse.nse import get_nse_opts

dfs = [get_nse_opts(s) for s in tqdm.tqdm(df_syms.symbol)]

100%|██████████| 195/195 [03:05<00:00,  1.05it/s]


In [28]:
# assemble the chains
import pandas as pd
df_chains = pd.concat(dfs,ignore_index=True)

In [40]:
# store the df_chains
from pathlib import Path
src_path = Path("C:/Users/kashi/python/ib/src")

In [46]:
data_path = src_path.parent.joinpath('data', 'master', 'df_chains.pkl')
df_chains.to_pickle(data_path)

# Get accurate dte

In [47]:
# get the dtes for the chains
from support import get_dte

exchange = df_syms.exchange.iloc[0]

dte = df_chains.expiry.apply(lambda x: get_dte(x, exchange))
dte.name = 'dte'

# insert the dtes
try:
    df_chains.insert(5, 'dte', dte)
except ValueError:
    log.warning('dte already in df_chains, will be refreshed')
    df_chains = df_chains.assign(dte=dte)

2023-03-11 12:19:42,437 | ib_log | dte already in df_chains, will be refreshed
